```markdown
# Classificação Multi-label com Classifier Chain

Este notebook apresenta uma introdução prática à **Classificação Multi-label (Multi-rótulo)**, um cenário onde um único exemplo de dado pode pertencer a várias classes simultaneamente. Utilizando uma base de dados musical, demonstraremos como preparar os dados, normalizar as características e treinar um modelo encadeado para prever múltiplos rótulos de forma eficiente.

### Objetivos de Aprendizagem
* Compreender a diferença entre classificação tradicional (multiclasse) e classificação multi-label.
* Aprender a estruturar e separar variáveis preditoras (atributos) de múltiplos rótulos (classes).
* Entender o papel da padronização de dados (`StandardScaler`) no desempenho de algoritmos de Machine Learning.
* Conhecer o conceito de **Classifier Chain** (Cadeia de Classificadores) e como ele lida com a dependência entre rótulos.
* Avaliar o modelo utilizando a métrica **Hamming Loss**.
```

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import hamming_loss
from sklearn.multioutput import ClassifierChain
from sklearn.preprocessing import StandardScaler

```markdown
### Passo 1: Importação das Bibliotecas e Ferramentas

**O que este passo faz:** Importa todos os pacotes necessários para manipulação de dados, divisão de bases, pré-processamento, modelagem e avaliação.

**Por que ele é necessário?**
* `pandas`: Essencial para carregar e manipular tabelas de dados (DataFrames).
* `train_test_split`: Divide nossos dados para que possamos testar o modelo em dados que ele nunca viu antes.
* `SVC` (Support Vector Classifier): O algoritmo de classificação baseado em Máquinas de Vetores de Suporte que servirá de base para o nosso classificador final.
* `hamming_loss`: A métrica ideal para avaliar classificação multi-label, medindo a fração de rótulos previstos incorretamente.
* `ClassifierChain`: Uma estratégia que treina múltiplos classificadores em cadeia, permitindo que a previsão de um rótulo influencie a previsão do próximo.
* `StandardScaler`: Normaliza nossos dados para que variáveis em escalas diferentes não distorçam o aprendizado do modelo.
```

In [ ]:
musica = pd.read_csv("Musica.csv")
musica.head(10)

```markdown
### Passo 2: Carregamento e Visualização Inicial dos Dados

**O que este passo faz:** Carrega o arquivo `Musica.csv` e exibe as 10 primeiras linhas da tabela.

**Por que ele é necessário?** Antes de escrever qualquer código de modelagem, precisamos conhecer a "cara" dos nossos dados: quais são as colunas, que tipo de informação elas guardam e se o arquivo foi carregado corretamente.

**Como interpretar a saída:** Você verá as colunas de rótulos (provavelmente emoções ou gêneros musicais no início) e as colunas de atributos técnicos da música (frequências, ritmos, etc.). Cada linha representa uma música diferente.
```

In [ ]:
musica.shape

```markdown
### Passo 3: Verificação das Dimensões do Dataset

**O que este passo faz:** Exibe a quantidade total de linhas (amostras) e colunas (atributos + rótulos) do conjunto de dados.

**Por que ele é necessário?** Saber o tamanho do dataset nos ajuda a entender a complexidade do problema e a validar se a divisão manual de colunas nos próximos passos estará matematicamente correta.
```

In [ ]:
classe = musica.iloc[:, 0:6].values
previsores = musica.iloc[:, 7:78].values

```markdown
### Passo 4: Divisão entre Variáveis Alvo (Classe) e Preditoras (Previsores)

**O que este passo faz:** Divide o DataFrame em dois arrays NumPy: `classe` (contendo os múltiplos rótulos que queremos prever) e `previsores` (as características técnicas das músicas usadas para fazer a previsão).

**Por que ele é necessário?** Algoritmos de Machine Learning precisam receber explicitamente o que é a "pergunta" (previsores) e o que é a "resposta" (classe).

💡 **Para pensar:** Note que o código seleciona as colunas de index `0:6` para classe e `7:78` para previsores. Isso significa que a coluna de índice `6` ficou de fora da divisão. Vale a pena verificar se isso foi intencional (por exemplo, se a coluna 6 for um ID ou texto não utilizado) ou se é uma inconsistência no mapeamento das colunas do dataset. `[VERIFICAR]`
```

In [ ]:
scaler = StandardScaler()
previsores = scaler.fit_transform(previsores)

```markdown
### Passo 5: Padronização das Características (Escalonamento)

**O que este passo faz:** Transforma os atributos para que todos tenham média 0 e desvio padrão 1.

**Por que ele é necessário?** Algoritmos como o SVM são extremamente sensíveis à escala dos dados. Se um atributo varia de 0 a 1 e outro varia de 0 a 1000, o modelo pode erroneamente dar muito mais importância ao segundo atributo apenas por causa de sua magnitude numérica. A padronização coloca todos os atributos em pé de igualdade.
```

In [ ]:
X_treinamento, X_teste, y_treinamento, y_teste =
  train_test_split(previsores, classe, test_size=0.3, random_state=0)

```markdown
### Passo 6: Divisão em Treino e Teste

**O que este passo faz:** Separa os dados em dois subconjuntos: 70% para o treinamento do modelo e 30% para a avaliação (teste).

**Por que ele é necessário?** Se avaliarmos o modelo usando os mesmos dados que ele usou para aprender, não saberemos se ele realmente aprendeu a generalizar ou se apenas decorou as respostas (overfitting). O teste simula o comportamento do modelo no mundo real com novos dados.
```

In [ ]:
svm = SVC()
classifier_chain = ClassifierChain(base_estimator=svm, random_state=0)
classifier_chain.fit(X_treinamento, y_treinamento)

```markdown
### Passo 7: Criação e Treinamento do Modelo (Classifier Chain)

**O que este passo faz:** Instancia um classificador SVM base, encapsula-o em uma estrutura de Cadeia de Classificadores (`ClassifierChain`) e treina o conjunto com os dados de treinamento.

**Por que ele é necessário?** Na classificação multi-label tradicional (como Binary Relevance), cada rótulo é previsto de forma totalmente independente. Contudo, em músicas, os rótulos costumam ser correlacionados (ex: uma música "triste" dificilmente será classificada como "festiva"). O **Classifier Chain** resolve isso organizando os classificadores em cadeia: a previsão do primeiro rótulo serve como característica adicional para a previsão do segundo, e assim por diante, respeitando as correlações entre as classes.
```

In [ ]:
y_pred = classifier_chain.predict(X_teste)
loss = hamming_loss(y_teste, y_pred)
print(f"Hamming Loss: {loss}")

```markdown
### Passo 8: Previsão e Avaliação com Hamming Loss

**O que este passo faz:** Faz as previsões para o conjunto de teste e calcula o **Hamming Loss** do modelo.

**Por que ele é necessário?** Precisamos medir numericamente a qualidade do modelo. O **Hamming Loss** calcula a fração de rótulos que foram classificados incorretamente.

**Como interpretar o resultado:** O Hamming Loss varia entre 0 e 1. Quanto **menor** o valor (mais próximo de 0), melhor é o desempenho do modelo, pois significa que ele errou menos rótulos individuais.
```

```markdown
---

## Diagnósticos, Dicas e Resumo Final

### Erros comuns e dicas úteis ao rodar este código
1. **Arquivo Não Encontrado (`FileNotFoundError`):** Garanta que o arquivo `Musica.csv` foi importado corretamente para a sessão local do seu Google Colab antes de executar a célula 2.
2. **Incompatibilidade de Formatos:** O `ClassifierChain` espera que as matrizes de classe (`y`) sejam estritamente binárias (0 ou 1). Caso existam valores contínuos ou nulos na variável `classe`, o treinamento falhará.
3. **Demora no Treinamento:** O algoritmo SVM pode ser lento em bases de dados muito grandes, especialmente quando envelopado em cadeias de classificadores. Caso o processamento trave, considere limitar o tamanho da amostra.

### Resumo do Aprendizado
* Vimos que a classificação **multi-label** difere da classificação tradicional por permitir múltiplos rótulos por registro.
* Aprendemos que o **escalonamento de dados** protege modelos baseados em distância (como SVM) de distorções de escala.
* Entendemos o funcionamento do **Classifier Chain** como uma solução inteligente para prever rótulos que possuem dependência mútua.
* Vimos que o **Hamming Loss** é uma métrica direta para avaliar erros individuais por rótulo.

---

## Exercícios Propostos (Pratique!)

1. **[Fácil]** Altere a proporção da divisão de treino/teste para 80/20 (usando `test_size=0.2` no passo 6) e compare se o valor de Hamming Loss melhorou ou piorou.
2. **[Médio]** Substitua o classificador base `SVC` por uma Árvore de Decisão (`DecisionTreeClassifier` do `sklearn.tree`) dentro do `ClassifierChain`. Meça o novo Hamming Loss e compare o tempo de execução.
3. **[Desafiador]** Implemente uma estratégia alternativa chamada **Binary Relevance** (usando `MultiOutputClassifier` do scikit-learn) com o mesmo estimador básico e compare o Hamming Loss obtido com o do `ClassifierChain`. Qual técnica se saiu melhor e por quê?

---

## Para ir além
* **Ordem da Cadeia:** No `ClassifierChain`, a ordem em que os rótulos são previstos altera o resultado final. Pesquise sobre cadeias com ordenação aleatória ou otimizada (`ClassifierChain(..., order='random')`).
* **Métricas de Avaliação Multi-label:** Pesquise sobre outras métricas importantes para este cenário, como *Subset Accuracy* (acerto exato de todos os rótulos simultaneamente) e as variações de F1-Score (*micro* e *macro*).
```